Обучите **простую рекуррентную нейронную сеть** (без GRU/LSTM, без внимания) **решать задачу дешифровки** [шифра Цезаря](https://ru.wikipedia.org/wiki/%D0%A8%D0%B8%D1%84%D1%80_%D0%A6%D0%B5%D0%B7%D0%B0%D1%80%D1%8F):
1. Написать алгоритм шифра Цезаря для генерации выборки (сдвиг на N каждой буквы). Например если N=2, то буква А переходит в букву С. Можно поиграться с языком на выбор (немецкий, русский и т.д.)
2. Создать архитектуру рекуррентной нейронной сети.
3. Обучить ее (вход - зашифрованная фраза, выход - дешифрованная фраза).
4. Проверить качество модели.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import re

In [2]:
with open('text.txt') as file:
    text = file.read()

Нормализуем текст:

In [3]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text) #оставляем только буквы и пробелы
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

text = normalize_text(text)

Готовим функцию для шифрования:

In [4]:
alphabet = "абвгдеёжзийклмнопрстуфхцчшщъыьэюя "
index_to_char = [w for w in alphabet]
char_to_index = {w: i for i, w in enumerate(alphabet)}

def encrypt(text, shift):
    encrypted_text = ''
    for char in text:
        encrypted_text += index_to_char[(char_to_index[char] + shift) % len(alphabet)]
    return encrypted_text

Шифруем текст:

In [5]:
encrypted_text = encrypt(text, 8)

Нарезаем текст на чанки для обучения:

In [6]:
def get_text_chunks(text, chunk_length):
    limit = len(text) - chunk_length + 1
    chunks = []
    for i in range(limit):
        chunk = text[i : i + chunk_length]
        chunks.append(chunk)
        
    return chunks

encrypted_chunks = get_text_chunks(encrypted_text, 20)
text_chunks = get_text_chunks(text, 20)

Создаем тензоры и заполняем их закодированным текстом

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X = torch.zeros((len(text_chunks), 20), dtype=int)
y = torch.zeros((len(text_chunks), 20), dtype=int)

for i in range(len(encrypted_chunks)):
    for j, w in enumerate(encrypted_chunks[i]):
        if j >= 20:
            break
        X[i, j] = char_to_index[w]

for i in range(len(text_chunks)):
    for j, w in enumerate(text_chunks[i]):
        if j >= 20:
            break
        y[i, j] = char_to_index[w]

Создаем класс нейронной сети:

In [8]:
vocab_size = len(alphabet)

class CaesarRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=30, hidden_dim=128):
        super(CaesarRNN, self).__init__()
        # Слой эмбеддингов: переводит индексы в векторы
        self.embedding = nn.Embedding(vocab_size, embedding_dim) 

        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        # Полносвязный слой из состояния RNN обратно в размер словаря
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x) 
        
        out, _ = self.rnn(embedded)
        out = self.fc(out) 

        return out



Создаем модель и обучаем ее:

In [9]:
model = CaesarRNN(vocab_size=vocab_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

epochs = 500


model.train()

for epoch in range(epochs):
    X_batch, Y_batch = X.to(device), y.to(device)

    optimizer.zero_grad()
    
    answers = model(X_batch)
    
    answers_flat = answers.view(-1, vocab_size)
    Y_batch_flat = Y_batch.view(-1)
    
    loss = criterion(answers_flat, Y_batch_flat)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

Epoch 100/500, Loss: 0.0014
Epoch 200/500, Loss: 0.0007
Epoch 300/500, Loss: 0.0004
Epoch 400/500, Loss: 0.0003
Epoch 500/500, Loss: 0.0002


Тестируем:

In [10]:
model.eval()
test_phrase = "привет как твои дела в мире глубокого обучения мой маленький друг"
encrypted_test = encrypt(test_phrase, shift=8)


test_input = torch.tensor([[char_to_index[c] for c in encrypted_test]]).to(device)
with torch.no_grad():
    pred_logits = model(test_input)
    pred_indices = torch.argmax(pred_logits, dim=2).squeeze().cpu().numpy()

predicted_text = "".join([index_to_char[idx] for idx in pred_indices])

correct = sum(1 for p, o in zip(predicted_text, test_phrase) if p == o)
accuracy = (correct / len(test_phrase)) * 100

print(f"Зашифровано:  {encrypted_test}")
print(f"Предсказано:  {predicted_text}")
print(f"Оригинал:     {test_phrase}")
print(f"Точность:     {accuracy:.2f}%")

Зашифровано:  чшрймъжтзтжъйцржлмузжйжфршмжкуыицтцкцжциыямхрёжфцсжфзумхгтрсжлшык
Предсказано:  привет как твои дела в мире глубокого обучения мой маленький друг
Оригинал:     привет как твои дела в мире глубокого обучения мой маленький друг
Точность:     100.00%
